<a href="https://colab.research.google.com/github/ml-project-sv/Cross-Platform-Binary-Code-Similarity-Detection/blob/main/model_experiment_feature-aggregator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
import os, glob, json
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

In [95]:
PATH = '/content/Cross-Platform-Binary-Code-Similarity-Detection'

# Wandb Login

In [96]:
import wandb
wandb.login()

True

# Arch-Opt Parsing Function

In [97]:
_ARCHES = ('x86', 'arm32', 'arm', 'mips32', 'mips')
_OPTS   = ('O0', 'O1', 'O2', 'O3')

In [98]:
def parse_arch_opt(src):
  arch = next((a for a in _ARCHES if f'-{a}-' in src), None)
  opt  = next((o for o in _OPTS   if f'-{o}' in src), None)
  return arch, opt

# Load Dataset

In [99]:
def load_dataset(dirs, min_blocks=4, n_feat=8):
  if isinstance(dirs, str):
    dirs = [dirs]

  files = []
  for d in dirs:
    files += glob.glob(os.path.join(d, '*.json'))

  if not files:
    raise FileNotFoundError(f'no .json under {dirs}')

  funcs, fnames, dropped = [], [], 0
  for fp in files:
    with open(fp) as fh:
      for line in fh:
        line = line.strip()
        if not line: continue

        r = json.loads(line)
        feats = r['features']
        if len(feats) < min_blocks:
          dropped += 1
          continue

        r['X'] = np.asarray(feats, dtype=np.float32)[:, :n_feat]
        r['succs'] = r['succs'] if 'succs' in r else None
        r['arch'], r['opt'] = parse_arch_opt(r.get('src', ''))

        funcs.append(r)
        fnames.append(r['fname'])

  uniq = sorted(set(fnames))
  classes = {n: i for i, n in enumerate(uniq)}
  labels = np.array([classes[n] for n in fnames], dtype=np.int64)
  print(f'[load] files={len(files)} funcs={len(funcs)} classes={len(uniq)} dropped(<{min_blocks} blocks)={dropped} n_feat={n_feat}')

  return funcs, labels, classes


# Split Dataset

In [100]:
def make_split(labels, ratios=(.8, .1, .1), seed=1337):
  n_cls = labels.max() + 1
  rng = np.random.default_rng(seed)
  cls_perm = rng.permutation(n_cls)

  n_train = int(ratios[0] * n_cls)
  n_val   = int(ratios[1] * n_cls)

  train_cls = set(cls_perm[:n_train].tolist())
  val_cls   = set(cls_perm[n_train: n_train + n_val].tolist())
  test_cls  = set(cls_perm[n_train + n_val:].tolist())


  train_mask = np.array([c in train_cls for c in labels])
  val_mask   = np.array([c in val_cls   for c in labels])
  test_mask  = np.array([c in test_cls  for c in labels])

  print(f'[split] train={train_mask.sum()} val={val_mask.sum()} test={test_mask.sum()}')
  return train_mask, val_mask, test_mask

# Functions for Metrics

In [101]:
def _normalize(V, eps=1e-10):
  return V / (np.linalg.norm(V, axis=1, keepdims=True) + eps)

In [102]:
def _group_by_class(labels):
  d = {}
  for i, c in enumerate(labels):
    d.setdefault(int(c), []).append(i)
  return d

In [103]:
# axis query and target belong to
def _axis(qa, qo, ta, to):
  if qa == ta and qo != to: return 'XO'
  if qa != ta and qo == to: return 'XA'
  if qa != ta and qo != to: return 'XM'
  return 'same'

# Metrics

In [104]:
def auc_pairs(E, score_fn, labels, n_pairs=8000, seed=1337):
  rng = np.random.default_rng(seed);
  by = _group_by_class(labels)

  # leave functions which have at least two compiled versions (eligible for positive pairs)
  pos_c = [c for c, v in by.items() if len(v) >= 2]

  s, y = [], []

  # same functions with different arch-opt pair
  for _ in range(n_pairs // 2):
    c = pos_c[rng.integers(len(pos_c))]
    i, j = rng.choice(by[c], 2, replace=False)
    s.append(float(score_fn(E, i, j)))
    y.append(1)

  # different functions
  n = len(labels); got = 0
  while got < n_pairs // 2:
    i, j = rng.integers(n), rng.integers(n)
    if labels[i] == labels[j]:
      continue
    s.append(float(score_fn(E, i, j)))
    y.append(0)
    got += 1

  return s, y

In [105]:
def retrieval(E, score_fn, labels, arches, opts, pool_sizes=(10, 100, 1000), n_queries=1000, ks=(1, 5, 10), seed=1337, per_axis=True):
  rng = np.random.default_rng(seed)
  by = _group_by_class(labels)

  pos_c = [c for c, v in by.items() if len(v) >= 2]
  axes = ['XO', 'XA', 'XM'] if per_axis else []
  out = {}

  for pool in pool_sizes:
    if pool > len(labels): continue

    # init recall hit for each top k
    rec = {k: 0 for k in ks}

    # init running sum of 1/rank
    mrr = 0.0

    # init number of queries processed
    Q = 0

    # init same metrics but for each axis now
    arec = {ax: {k: 0 for k in ks} for ax in axes}
    amrr = {ax: 0.0 for ax in axes}
    acnt = {ax: 0 for ax in axes}


    for _ in range(n_queries):
      # choose class and positive pairs of functions
      c = pos_c[rng.integers(len(pos_c))]
      q, tgt = rng.choice(by[c], 2, replace=False)

      # choosing functions with distinct classes
      dist = []
      while len(dist) < pool - 1:
        d = rng.integers(len(labels))
        if labels[d] != c:
          dist.append(d)

      cand = np.array([tgt] + dist)
      sims = score_fn(E, q, cand)

      # calculate where positive pair is ranked
      rank = np.where(np.argsort(-sims) == 0)[0][0] + 1

      # update metrics
      mrr += 1.0 / rank
      for k in ks:
        if rank <= k: rec[k] += 1
      Q += 1

      # if per_axis is enabled, log same metrics but separately for each (q, tgt) axis
      if per_axis:
        ax = _axis(arches[q], opts[q], arches[tgt], opts[tgt])
        if ax in arec:
          amrr[ax] += 1.0 / rank; acnt[ax] += 1
          for k in ks:
            if rank <= k: arec[ax][k] += 1

    # save results to out
    for k in ks:
      out[f'recall@{k}_pool{pool}'] = rec[k] / Q
    out[f'mrr_pool{pool}'] = mrr / Q

    for ax in axes:
      if acnt[ax] == 0:
        continue
      for k in ks:
        out[f'{ax}_recall@{k}_pool{pool}'] = arec[ax][k] / acnt[ax]
      out[f'{ax}_mrr_pool{pool}'] = amrr[ax] / acnt[ax]

  return out

# Evaluate

In [106]:
def evaluate(embedder, score_fn, funcs, labels, pool_sizes=(10, 100, 1000), n_pairs=8000, n_queries=500, ks=(1, 5, 10), per_axis=True):
  E = embedder.embed_batch(funcs)
  arches = [f.get('arch') for f in funcs]
  opts   = [f.get('opt')  for f in funcs]

  # roc and pr
  s, y = auc_pairs(E, score_fn, labels, n_pairs=n_pairs)
  roc_auc, pr_auc = roc_auc_score(y, s), average_precision_score(y, s)

  # retrieval
  retr = retrieval(E, score_fn, labels, arches, opts, pool_sizes=pool_sizes, n_queries=n_queries, ks=ks, per_axis=per_axis)

  return  {'roc_auc': roc_auc, 'pr_auc': pr_auc, **retr}

# Wandb Logging

In [107]:
def _cast(v):
    return float(v) if isinstance(v, (np.floating, np.integer, np.bool_)) else v

def log(run, results, table_name='metrics'):
    for tag, res in results.items():
        payload = {f'{tag}/{k}': _cast(v) for k, v in res.items()}
        run.log(payload)
        for k, v in payload.items():
            run.summary[k] = v

    table = wandb.Table(columns=['eval_set', 'metric', 'value'])
    for tag, res in results.items():
        for k, v in res.items():
            table.add_data(tag, k, _cast(v))
    run.log({table_name: table})

# Model

In [112]:
class FeatureAggregator():
  def embed_batch(self, funcs):
    out = []
    for r in funcs:
      X = r['X']
      out.append(np.concatenate([X.mean(0), X.sum(0), X.max(0), X.std(0), [len(X)]]))
    E = np.asarray(out, dtype=np.float64)
    return (E - E.mean(0)) / (E.std(0) + 1e-8)

def cos_sim(E, query_index, cand_index):
  En = _normalize(E)
  return np.dot(En[query_index], En[cand_index].T)

 # Read Dataset

In [109]:
min_blocks = 4; n_feat = 7

openssl_funcs, openssl_labels, openssl_classes = load_dataset(f'{PATH}/data_acfg/openssl_1_0_1f_acfg', min_blocks=min_blocks, n_feat=n_feat)
sqlite_funcs,  sqlite_labels,  sqlite_classes  = load_dataset(f'{PATH}/data_acfg/sqlite3_acfg',        min_blocks=min_blocks, n_feat=n_feat)
zlib_funcs,    zlib_labels,    zlib_classes    = load_dataset(f'{PATH}/data_acfg/zlib_acfg',           min_blocks=min_blocks, n_feat=n_feat)

openssl_train_mask, openssl_val_mask, openssl_test_mask = make_split(openssl_labels, ratios=(.8, .1, .1))
_,                  _,                zlib_test_mask    = make_split(zlib_labels,    ratios=(0, 0, 1))
_,                  _,                sqlite_test_mask  = make_split(sqlite_labels,  ratios=(0, 0, 1))

openssl_test_funcs = [f for f, m in zip(openssl_funcs, openssl_test_mask) if m]
zlib_test_funcs    = [f for f, m in zip(zlib_funcs,    zlib_test_mask)    if m]
sqlite_test_funcs  = [f for f, m in zip(sqlite_funcs,  sqlite_test_mask)  if m]

openssl_test_labels = openssl_labels[openssl_test_mask]
zlib_test_labels    = zlib_labels[zlib_test_mask]
sqlite_test_labels  = sqlite_labels[sqlite_test_mask]

[load] files=12 funcs=40624 classes=4332 dropped(<4 blocks)=30344 n_feat=7
[load] files=12 funcs=19864 classes=3679 dropped(<4 blocks)=5773 n_feat=7
[load] files=12 funcs=3093 classes=368 dropped(<4 blocks)=941 n_feat=7
[split] train=32475 val=4074 test=4075
[split] train=0 val=0 test=3093
[split] train=0 val=0 test=19864


# Train Model

In [110]:
# Training not needed for this model
model = FeatureAggregator()

# Test model

In [113]:
pool_sizes=(10, 100, 1000); n_pairs=8000; n_queries=1000; ks=(1, 5, 10); per_axis=True

datasets = {
    'openssl_1_0_1f': (openssl_test_funcs, openssl_test_labels),
    'zlib-1.3.1':     (zlib_test_funcs,    zlib_test_labels),
    'sqlite3':        (sqlite_test_funcs,  sqlite_test_labels),
}

run = wandb.init(
    project='binsim_baseline_feature_aggregator',
    name=f'baseline_feature_aggregator__n_feat_{n_feat}__min_blocks_{min_blocks}',
    config = {
        'model': 'aggregate_baseline', 'n_feat': n_feat, 'min_blocks': min_blocks,
        'pool_sizes': pool_sizes, 'n_pairs': n_pairs,
        'n_queries': n_queries, 'ks': ks, 'per_axis': per_axis, 'seed': 1337,
        'arches': ['x86', 'arm32', 'mips32'], 'opts': ['O0', 'O1', 'O2', 'O3'],
    }
)

results = {}

for tag, (test, labels) in datasets.items():
  results[tag] = evaluate(model, cos_sim, test, labels, pool_sizes=pool_sizes, n_pairs=n_pairs, n_queries=n_queries, ks=ks, per_axis=per_axis)

log(run, results)
run.finish()